**Required imports**

In [ ]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
import numpy as np
import joblib

**Loading Dataset into pandas dataframe from hugging face**

In [ ]:
ds = load_dataset("milistu/AMAZON-Products-2023")
x_train=pd.DataFrame(ds['train'].select_columns(['title', 'description', 'main_category']))
x_train.head()

**Analyzing Data**

In [ ]:
print(f"The number of rows and columns are : {x_train.shape}")
print(f"There are {x_train['main_category'].nunique()} unique categories")
print(f"\nNull values\n{x_train.isnull().sum()}")

lol 24k null values in main_category feature

**predicting those 24k null values in categories_prediction.ipynb**

In [ ]:
print(f"\n{x_train['main_category'].value_counts().head(20)}")

In [ ]:
print(f"\n{x_train['main_category'].value_counts().tail(18)}")

**Loading main_category predicted cleaned data**

In [ ]:
cleaned_df=pd.read_csv('clean_data.csv')
cleaned_df.head()

In [ ]:
cleaned_df['main_category'].value_counts().tail(20)

analyzing and cleaning null data

In [ ]:
print(cleaned_df.isnull().sum())
cleaned_df=cleaned_df.dropna()
print(f"\n\nAfter droping\n\n{cleaned_df.isna().sum()}")

**converting the text into vector**

In [ ]:
t_tfidf=TfidfVectorizer(max_features=25000, ngram_range=(1,2), stop_words='english')
c_tfidf=TfidfVectorizer(max_features=50,stop_words='english')
d_tfidf=TfidfVectorizer(max_features=25000,ngram_range=(1,2),stop_words='english')

In [ ]:
t_vector=t_tfidf.fit_transform(cleaned_df['title'])
c_vector=c_tfidf.fit_transform(cleaned_df['main_category'])
d_vector=d_tfidf.fit_transform(cleaned_df['description'])

assigning weight to each column based on their importance

In [ ]:
tw=3.0
cw=2.0
dw=1.0

In [ ]:
t_vector=t_vector*tw
c_vector=c_vector*cw
d_vector=d_vector*dw

final entire vectorized sparse matrix

In [ ]:
s_v=sp.hstack([t_vector,c_vector,d_vector],format='csr')

**Function to calculate and return top similar product of query**

In [ ]:
def similarity_calculate(title="",category="",description="",top_similar=5):
    title=title.lower()
    category=category.lower()
    description=description.lower()
    title_vector=t_tfidf.transform([title])*tw
    category_vector=c_tfidf.transform([category])*cw
    description_vector=d_tfidf.transform([description])*dw
    final_vector=sp.hstack([title_vector,category_vector,description_vector],format='csr')
    similarity=cosine_similarity(final_vector,s_v).flatten()
    top_similar_indx=np.argsort(similarity)[::-1][:top_similar]
    recommended = cleaned_df.iloc[top_similar_indx].copy()
    return recommended[['title','main_category']]

**checking model output**

In [76]:
print(similarity_calculate("Nike shoes","fashion"))

                          title   main_category
76653      nike girls boyshorts  amazon fashion
80513               nike casual  amazon fashion
72717               nike modern  amazon fashion
109704             nike hoodies  amazon fashion
81902   nike unisexchild modern  amazon fashion
